In [1]:
import pandas as pd
from pathlib import Path
import os
import json

In [2]:
try:
    root = Path("/home/daniel/code/clasificador_selectividad")
    os.chdir(root)
except Exception as e:
    root = Path("/Users/daniel/code/clasificador_selectividad")
    os.chdir(root)

Obtener ejercicios de un tema

In [3]:
df_enun = pd.read_csv(root/"csv/ejercicios_clasificados.csv", encoding="utf-8")

In [4]:
TOPIC_TO_CLASSIFY = "geometría"

In [5]:
if TOPIC_TO_CLASSIFY:
    topic = TOPIC_TO_CLASSIFY.lower().strip().replace(" ", "_")
    df_enun[df_enun.tema == topic].to_csv(root / f"classify_using_api/df_{topic}.csv")
    df_to_classify = df_enun.loc[df_enun.tema == topic, ["id", "asignatura", "tema", "enunciado"]]

In [6]:
CSV_TO_CLASSIFY = root / f"classify_using_api/df_{topic}.csv"
JSON_PATH = root / f"classify_using_api/{topic}.json"
CSV_EXPORT_PATH = root / "csv/clasificados_tema"

In [7]:
with open(JSON_PATH, encoding="utf-8") as f:
    topic_dict = json.load(f)        # load → archivo

In [8]:
topic_dict = topic_dict[topic]
topic_dict

{'vectores': ['tetraedro',
  'triángulo',
  'área',
  'volumen',
  'vectores',
  'coplanario',
  'paralelogramo',
  'parelelepípedo',
  'vértices'],
 'ángulos': [' ángulo'],
 'segmento en partes iguales': ['partes iguales'],
 'planos': ['plano'],
 'intersección plano con ejes coordenados': ['con los ejes coordenados',
  'planos cartesianos'],
 'simétrico': ['simétrico'],
 'recta': [' recta'],
 'posición relativa': ['posición relativa',
  'perpendicular',
  ' corta',
  'paralelos',
  'paralela',
  'paralelo ',
  'se corten'],
 'distancias': ['distancia', 'equidista']}

In [9]:
df_to_classify.head()

,id,asignatura,tema,enunciado
2969,mates_ii_geometria_vectores_vectores_vectores_...,Mates II,geometría,"Considera los puntos A ( 1 , 2 , 0 ) , B ( 2 ,..."
2970,mates_ii_geometria_vectores_vectores_angulos_2...,Mates II,geometría,"Considera los puntos A ( 1 , 2 , 0 ) , B ( 2 ,..."
2971,mates_ii_geometria_vectores_angulos_vectores_2...,Mates II,geometría,"Considera los puntos A ( 1 , 2 , 0 ) , B ( 2 ,..."
2972,mates_ii_geometria_vectores_angulos_angulos_20...,Mates II,geometría,"Considera los puntos A ( 1 , 2 , 0 ) , B ( 2 ,..."
2973,mates_ii_geometria_angulos_vectores_vectores_2...,Mates II,geometría,"Considera los puntos A ( 1 , 2 , 0 ) , B ( 2 ,..."


In [ ]:
def classify_exercise(row, topic_dict = topic_dict):
    enun = row["enunciado"].lower()
    tipos_ejercicio = []
    if pd.notna(enun):
        for tipo, keywords in topic_dict.items():
            for kw in keywords:
                if kw in enun:
                    tipos_ejercicio.append(tipo)
                    break
        tipos_ejercicio = "|".join(tipos_ejercicio)

        return tipos_ejercicio
    return None

In [11]:
df_to_classify["tipo_ejercicio"] = pd.Series(index=df_to_classify.index, dtype="object")
df_to_classify["observaciones"] = pd.Series(index=df_to_classify.index, dtype="object")

In [12]:
df_to_classify["tipo_ejercicio"] = df_to_classify.apply(classify_exercise, axis=1)

In [13]:
df_to_classify.sample(3)

,id,asignatura,tema,enunciado,tipo_ejercicio,observaciones
12766,mates_ii_geometria_distancias_planos_recta_200...,Mates II,geometría,3x2y 0\nSea r la recta de ecuaciones r  ....,planos|recta|posición relativa|distancias,NaN
3324,mates_ii_geometria_planos_recta_recta_2024_jun...,Mates II,geometría,Considera las rectas r \n\n2 x −\ny\nz\n=\...,planos|recta|posición relativa|distancias,NaN
3216,mates_ii_geometria_planos_recta_posicion_relat...,Mates II,geometría,Considera las rectas r \nx\n1\n=\ny\n−\n−\n1\...,planos|recta|posición relativa,NaN


Ejercicios sin clasificar

In [14]:
# df_to_classify.tipo_ejercicio.value_counts()

In [ ]:
# df_to_classify.to_csv(CSV_EXPORT_PATH / f"ejercicios_clasificados_{topic}.csv", index = False, encoding="utf-8")